In [ ]:
import pandas as pd
import numpy as np
import cvxpy as cp
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf

## 1. データの取得

In [ ]:
tickers = [
    "VOO",   # 海外安定
    "ARKK",  # 海外新興
    "1306.T",# 国内安定
    "2516.T",# 国内新興
    "TLT"    # 安全資産
]

fx_ticker = "USDJPY=X"

In [ ]:
# 過去5年分の終値をまとめて取得

all_tickers = tickers + [fx_ticker]
df_data = yf.download(all_tickers, period="5y")["Close"]

In [ ]:
df_data.head(10)

In [ ]:
# yfinanceは、裏でbeatifulsopuが動いててスクレイピングをしているので、
# 一度取得したデータを保存しておく
# 理由①yahoo financeのサーバーからブロックされないように
# 理由②yahoo financeのページの要素が変わったら、エラーなどでデータ取得できなくなる可能性があるため

df_data.to_csv("./data/data.csv", encoding="utf-8")

In [ ]:
# yfinanceからダウンロードした場合、一旦indexをリセットする
df_data.columns.name = None
df_data = df_data.reset_index(names="Date")

## 2. データの中身の確認

In [ ]:
# notebookをrestartした場合などは、すでに保存済みのdata.csvを読み込む

if "df_data" in locals():
    print("df_data already exits")
else:
    df_data = pd.read_csv("./data/data.csv", encoding="utf-8")

In [ ]:
df_data.head(5)

In [ ]:
df_data.shape

### 2.1 欠損値確認

In [ ]:
df_data.isnull().sum()

# ↓結果
# Date         0
# 1306.T      80
# 2516.T      81
# ARKK        49
# TLT         49
# USDJPY=X     3
# VOO         49
# Dateは欠損なし。その他のカラムは多少あるが、1303行のうちの欠損なので、10％に満たない
# そのため、欠損は補完する


In [ ]:
df_data = df_data.ffill()

# 補完方法は、前日の値をそのまま引き継ぐ（前方補間 / forward fill）

### 2.2 円建て換算

In [ ]:
# 米国市場の株価はドルなので、比較をしやすいように日本円に変換する

usa_tickers = ["VOO", "ARKK", "TLT"]
jpy_tickers = ["1306.T", "2516.T"]

# 日本株はそのままコピー

df_jpy = df_data[jpy_tickers].copy()

# 米国株はドル円の値をかける
for t in usa_tickers:
    df_jpy[t] = df_data[t] * df_data[fx_ticker]


In [ ]:
# 「1口（1株）あたりの値段」
# すべて１口あたりの値段ではあるが、市場と銘柄ごとに１口あたりの料金が異なるため、値差があるように見える

df_jpy.head(5)

### 2.3 データの概要の確認

In [ ]:
df_jpy.describe()

In [ ]:
# 桁の影響を抑えて比較するために、変動係数を確認

df_jpy.describe().loc["std"] / df_jpy.describe().loc["mean"]

# ↓結果
# 1306.T    0.278334
# 2516.T    0.153682
# VOO       0.306762
# ARKK      0.331167
# TLT       0.053620

# TLTはやはり国債なので、安定している。
# 2516.Tの変動が株の中では比較的小さい

### 2.4 グラフで値動きを確認

In [ ]:
df_jpy.head(3)

In [ ]:
df_jpy["Date"] = df_data["Date"].copy()

In [ ]:
df_jpy

In [ ]:
# 桁が違うので、日本株と米株を別々で描画する

fig, axes = plt.subplots(2, 1, figsize=(6, 8), sharex=True)

# 上のグラフ（ax[0]）に系列1をプロット
for ticker in jpy_tickers:
    axes[0].plot(df_jpy["Date"], df_jpy[ticker], label=ticker)
axes[0].set_title("Japanese tickers")
axes[0].legend() 

# 下のグラフ（ax[1]）に系列2をプロット
for ticker in usa_tickers:
    axes[1].plot(df_jpy["Date"], df_jpy[ticker], label=ticker)
axes[1].set_title("American tickers")
axes[1].legend()  # 凡例を表示

plt.xlabel("Date")
plt.ylabel("Price [JPY]")
plt.tight_layout()  # タイトルやラベルの被りを防ぐ
plt.show()

## 3. ポートフォリオの分配の最適化

In [ ]:
df_optimisation = df_jpy.copy()

### 3.1 初日の価格を基準に割合になおして、条件を合わせる

In [ ]:
# 桁の異なる銘柄を比較しやすくするため、初日の価格を1とした指数化を行う。
# なお、後続の前日比計算だけが目的であれば、この処理は不要。

for col in df_optimisation.drop("Date", axis=1).columns:
    
    col_name = f"{col}_ratio"

    # 銘柄ごとに桁が違うので、初日を基準として割合に変換する
    df_optimisation[col_name] = df_optimisation[col]/df_optimisation[col][0]

In [ ]:
df_optimisation.head(3)

### 3.2 前日比を求める

In [ ]:
df_DoD = df_optimisation[[i + "_ratio" for i in tickers]].pct_change().dropna()


#### 3.2.1 前日比の相関係数のヒートマップ

In [ ]:
df_DoD.head(3)

In [ ]:
correlation_matrix = df_DoD.corr()

In [ ]:
# 相関係数のヒートマップ

plt.figure(figsize=(6, 4))  # グラフのサイズを設定
sns.heatmap(
    correlation_matrix, 
    annot=True,             # マス目に数値を表示
    cmap="coolwarm",        # 青＝負の相関、赤＝正の相関
    vmin=-1, vmax=1         # カラーバーの範囲設定
)

plt.title("Correlation Matrix Heatmap")
plt.show()

### 3.3　最適化

In [ ]:
# 5年間の平均リターンと、銘柄同士の連動性（共分散行列）を年率換算（×252営業日）
mean_returns = df_DoD.mean().values * 252
cov_matrix = df_DoD.cov().values * 252
tickers_DoD = df_DoD.columns.tolist()

In [ ]:
n_assets = len(tickers_DoD)
weights = cp.Variable(n_assets) # 最適化の比率。

In [ ]:
# 目的関数の設定
# 分散投資＋安定投資を目的する

# 目的関数はポートフォリオ全体の「リスク（分散）」を最小化する

# portfolio_variance = cp.quad_form(weights, cov_matrix)
# objective = cp.Minimize(portfolio_variance)
# →これだけだと、値動きがほぼないTLTになってしまうので、分散だけではなく、利益も求める式に変える
# 金融業界の定番：平均・分散の目的関数を活用する
# → Max: (期待リターン) - γ*(ポートフォリオの分散)

# ポートフォリオの期待リターンに重みパラメータをかける
portfolio_return = mean_returns @ weights

# 分散をリスクとして定義する
portfolio_variance = cp.quad_form(weights, cov_matrix)

# リターンとリスクのどちらを重視するかを調整するパラメータ
# 今回は実験用に5.0を設定
gamma = 5.0

# 目的関数：(リターン - リスクペナルティ) を最大化する
objective = cp.Maximize(portfolio_return - gamma * portfolio_variance)

In [ ]:
# 制約条件の設定

constraints = [
    cp.sum(weights) == 1.0,   # 条件1: 投資比率の合計は100% (1.0)
    weights >= 0.10,          # 条件2: 分散投資の原則。各銘柄最低10%は買う
    weights <= 0.50           # 条件3: 特定の銘柄に依存しすぎないよう、最大でも50%まで
]

In [ ]:
# 最適化問題を解く
prob = cp.Problem(objective, constraints)
prob.solve()

In [ ]:
optimized_weights = weights.value

In [ ]:
# 可視化
plt.figure(figsize=(6, 6))
plt.pie(
    optimized_weights, 
    labels=tickers, 
    autopct="%1.1f%%", 
    startangle=140, 
    pctdistance=0.85,
    colors=["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd"]
)

In [ ]:
df_weights = pd.DataFrame({
    "Ticker": tickers,
    "Weight": optimized_weights
})

df_weights